1.
<p>Load 'bike_listings_s7.csv' in a new notebook 'bike_duplicates.ipynb'. With `pd.concat`, add 25 exact copies of random rows (`sample(25, random_state=42)`) and 10 relisted rows with the same listing_id but price_inr cut by 5%. Use `duplicated()` to count the exact copies (25) and `duplicated(subset='listing_id', keep=False)` to show every row sharing a listing_id. Hint: Missed Session 7? Any 400-row bike table with a unique listing_id works.</p>
2.
<p>Remove the duplicates in order: `drop_duplicates()` to get `bikes_nodup` (410 rows), then keep only the latest relisted row per listing_id with `drop_duplicates(subset='listing_id', keep='last')` into `bikes_unique`. Print the row count after each step and assert 400 rows with a unique listing_id. Hint: The relisted rows were appended last, so `keep='last'` keeps the new price.</p>
3.
<p>In a copy of `bikes_unique`, lower-case brand and city for 30 random rows, add trailing spaces to 20, and misspell Veloce as 'Velose' and Ahmedabad as 'Ahmdabad' in 12 rows each. Show `value_counts()` before fixing, then repair with `str.strip()`, `str.title()`, `replace()` and `difflib.get_close_matches` (cutoff 0.8) until brand and city each have exactly 4 values.</p>
4.
<p>Add an `is_insured` column with messy values (Yes, yes, Y, NO, no, N, ' yes ') to the repaired frame and unify it to 1/0 with `str.strip()`, `str.lower()` and `map()`. Check 400 unique listing_ids, 4 brands, 4 cities and only 0 or 1 in `is_insured`, then save 'bike_listings_clean.csv'. Run Restart & Run All and push to 's08_bike_duplicates' in your GitHub repo 'ds_analysis_portfolio'.</p>

In [1]:
import numpy as np
import pandas as pd
from difflib import get_close_matches

In [2]:
bikes = pd.read_csv("bike_listings_s7.csv")

print(bikes.head())
print("Original rows:", len(bikes))
print("Columns:", bikes.columns.tolist())

   listing_id    brand      city  model_year  km_driven  owner_count  \
0         358   Tarang  Vadodara        2008    27522.0            2   
1         356   Sundar     Surat        2008    13485.0            3   
2         350   Sundar     Surat        2008    40258.0            3   
3         342   Tarang     Surat        2008    61676.0            1   
4          66  Rajhans  Vadodara        2008    28069.0            1   

   price_inr  price_capped  price_wins     km_log  is_first_owner   listed_on  \
0      51958       51958.0     51958.0  10.222777           False  2024-06-24   
1     102539      102539.0    102539.0   9.509407           False  2024-07-15   
2      90128       90128.0     90128.0  10.603089           False  2024-01-26   
3      50963       50963.0     50963.0  11.029666            True  2024-03-08   
4      50963       50963.0     50963.0  10.242457            True  2024-07-30   

   listed_month  
0             6  
1             7  
2             1  
3       

In [3]:
# Select 25 random rows and copy them exactly
exact_copies = bikes.sample(25, random_state=42)

# Select 10 rows for relisting
relisted = bikes.sample(10, random_state=10).copy()

# Reduce their prices by 5%
relisted["price_inr"] = relisted["price_inr"] * 0.95

# Combine all rows
bikes_with_dups = pd.concat(
    [bikes, exact_copies, relisted],
    ignore_index=True
)

print("Rows after adding duplicates:", len(bikes_with_dups))

Rows after adding duplicates: 435


In [4]:
exact_duplicate_count = bikes_with_dups.duplicated().sum()

print("Exact duplicate rows:", exact_duplicate_count)

Exact duplicate rows: 25


In [5]:
same_listing = bikes_with_dups[
    bikes_with_dups.duplicated(subset="listing_id", keep=False)
]

same_listing = same_listing.sort_values("listing_id")

print(same_listing[["listing_id", "price_inr"]])
print("Rows sharing listing IDs:", len(same_listing))

     listing_id  price_inr
266          11    99553.0
408          11    99553.0
409          32   112263.0
126          32   112263.0
205          62    97595.0
..          ...        ...
411         379   132435.0
404         386    69965.0
93          386    69965.0
42          395    60046.0
415         395    60046.0

[69 rows x 2 columns]
Rows sharing listing IDs: 69


In [6]:
bikes_nodup = bikes_with_dups.drop_duplicates()

print("Rows after removing exact duplicates:", len(bikes_nodup))

Rows after removing exact duplicates: 410


In [7]:
bikes_unique = bikes_nodup.drop_duplicates(
    subset="listing_id",
    keep="last"
)

print("Rows after removing repeated listing IDs:", len(bikes_unique))
print("Unique listing IDs:", bikes_unique["listing_id"].nunique())

assert len(bikes_unique) == 400
assert bikes_unique["listing_id"].is_unique

print("Successfully kept 400 unique listings!")

Rows after removing repeated listing IDs: 400
Unique listing IDs: 400
Successfully kept 400 unique listings!


In [8]:
bikes_clean = bikes_unique.copy()

# Convert brand and city to lowercase for 30 random rows
lower_rows = bikes_clean.sample(30, random_state=7).index

bikes_clean.loc[lower_rows, "brand"] = (
    bikes_clean.loc[lower_rows, "brand"].str.lower()
)

bikes_clean.loc[lower_rows, "city"] = (
    bikes_clean.loc[lower_rows, "city"].str.lower()
)

# Add trailing spaces to 20 random rows
space_rows = bikes_clean.sample(20, random_state=8).index

bikes_clean.loc[space_rows, "brand"] = (
    bikes_clean.loc[space_rows, "brand"].astype(str) + " "
)

bikes_clean.loc[space_rows, "city"] = (
    bikes_clean.loc[space_rows, "city"].astype(str) + " "
)

# Misspell Veloce in 12 rows
veloce_rows = bikes_clean[
    bikes_clean["brand"].astype(str).str.strip().str.title() == "Veloce"
].sample(12, random_state=9).index

bikes_clean.loc[veloce_rows, "brand"] = "Velose"

# Misspell Ahmedabad in 12 rows
ahmedabad_rows = bikes_clean[
    bikes_clean["city"].astype(str).str.strip().str.title() == "Ahmedabad"
].sample(12, random_state=10).index

bikes_clean.loc[ahmedabad_rows, "city"] = "Ahmdabad"

print("Messy values added successfully!")

Messy values added successfully!


In [9]:
print("Brand values before cleaning:")
print(bikes_clean["brand"].value_counts())

print("\nCity values before cleaning:")
print(bikes_clean["city"].value_counts())

Brand values before cleaning:
brand
Tarang      100
Sundar       90
Rajhans      77
Veloce       74
Velose       12
sundar        8
tarang        8
Tarang        7
rajhans       6
veloce        5
Sundar        5
Rajhans       3
Veloce        3
sundar        1
veloce        1
Name: count, dtype: int64

City values before cleaning:
city
Rajkot        103
Surat          87
Ahmedabad      82
Vadodara       68
Ahmdabad       12
vadodara       10
Surat           7
surat           7
ahmedabad       6
rajkot          5
Vadodara        4
Ahmedabad       4
Rajkot          3
vadodara        1
rajkot          1
Name: count, dtype: int64


In [10]:
bikes_clean["brand"] = (
    bikes_clean["brand"].str.strip().str.title()
)

bikes_clean["city"] = (
    bikes_clean["city"].str.strip().str.title()
)

print("Spaces and letter cases corrected!")

Spaces and letter cases corrected!


In [11]:
# Correct brand names
brand_options = ["Veloce", "Rajhans", "Sundar", "Tarang"]

brand_corrections = {}

for value in bikes_clean["brand"].unique():
    match = get_close_matches(
        value, brand_options, n=1, cutoff=0.8
    )

    if match:
        brand_corrections[value] = match[0]

bikes_clean["brand"] = bikes_clean["brand"].replace(
    brand_corrections
)

# Correct city names
city_options = ["Ahmedabad", "Surat", "Vadodara", "Rajkot"]

city_corrections = {}

for value in bikes_clean["city"].unique():
    match = get_close_matches(
        value, city_options, n=1, cutoff=0.8
    )

    if match:
        city_corrections[value] = match[0]

bikes_clean["city"] = bikes_clean["city"].replace(
    city_corrections
)

print("Spelling corrections completed!")

Spelling corrections completed!


In [12]:
print("Corrected brand values:")
print(bikes_clean["brand"].value_counts())

print("\nCorrected city values:")
print(bikes_clean["city"].value_counts())

print("\nTotal brands:", bikes_clean["brand"].nunique())
print("Total cities:", bikes_clean["city"].nunique())

Corrected brand values:
brand
Tarang     115
Sundar     104
Veloce      95
Rajhans     86
Name: count, dtype: int64

Corrected city values:
city
Rajkot       112
Ahmedabad    104
Surat        101
Vadodara      83
Name: count, dtype: int64

Total brands: 4
Total cities: 4


In [14]:
insurance_values = [
    "Yes", "yes", "Y", "NO", "no", "N", " yes "
]

np.random.seed(42)

bikes_clean["is_insured"] = np.random.choice(
    insurance_values,
    size=len(bikes_clean)
)

print("Insurance values before cleaning:")
print(bikes_clean["is_insured"].value_counts())

Insurance values before cleaning:
is_insured
NO       69
Yes      66
Y        60
N        58
 yes     57
yes      46
no       44
Name: count, dtype: int64


In [15]:
bikes_clean["is_insured"] = (
    bikes_clean["is_insured"]
    .str.strip()
    .str.lower()
    .map({
        "yes": 1,
        "y": 1,
        "no": 0,
        "n": 0
    })
)

print("Insurance values after cleaning:")
print(bikes_clean["is_insured"].value_counts())

Insurance values after cleaning:
is_insured
1    229
0    171
Name: count, dtype: int64


In [16]:
print("Total rows:", len(bikes_clean))
print("Unique listing IDs:", bikes_clean["listing_id"].nunique())
print("Total brands:", bikes_clean["brand"].nunique())
print("Total cities:", bikes_clean["city"].nunique())
print("Insurance values:", bikes_clean["is_insured"].unique())

# Check all conditions
assert len(bikes_clean) == 400
assert bikes_clean["listing_id"].is_unique
assert bikes_clean["brand"].nunique() == 4
assert bikes_clean["city"].nunique() == 4
assert set(bikes_clean["is_insured"].unique()) == {0, 1}

print("\nAll checks passed successfully!")

# Save final cleaned dataset
bikes_clean.to_csv("bike_listings_clean.csv", index=False)

print("bike_listings_clean.csv saved successfully!")

Total rows: 400
Unique listing IDs: 400
Total brands: 4
Total cities: 4
Insurance values: [1 0]

All checks passed successfully!
bike_listings_clean.csv saved successfully!
